# 03 · 100 Hz filtered ECG CNN

Run after Phase 1 and Phase 2. Notebook 01 saves the literature-derived subject-group split. This notebook consumes it and completes the filtered-waveform CNN needed for RQ2. It never changes the saved Phase 1 assignments or Phase 2 results. The earlier test records have already been inspected, so this is an exploratory evaluation rather than a pristine confirmatory test. The grouped IDs are reconstructed from published record pairings; original patient identifiers are unavailable.

Input contract: **100 Hz, 6,000 samples per minute**. The original 3,009-parameter architecture, including internal striding/pooling, is unchanged. Fresh training is required; 25 Hz checkpoints cannot be reused. Unchanged kernels span shorter durations in seconds at 100 Hz. This single-split notebook is a reference; Notebook 07 defines the new portfolio search protocol.

In [ ]:
import hashlib
import json
import os
import random
import tempfile
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
PHASE1 = Path(os.environ.get("APNEA_RUN", str(ROOT / "data/processed/phase1")))
SPLIT_RUN = Path(os.environ.get("APNEA_SPLIT_RUN", str(ROOT / "data/processed/phase1-grouped")))
if not SPLIT_RUN.is_absolute():
    SPLIT_RUN = ROOT / SPLIT_RUN
PHASE3 = Path(
    os.environ.get(
        "APNEA_PHASE3_RUN", str(ROOT / "data/processed/phase3-100hz-v2/standalone-20261108")
    )
)
SEED, FS_IN, FS_MODEL, WINDOW_SAMPLES = 67, 100, 100, 6000
torch.set_num_threads(min(4, os.cpu_count() or 1))
if not PHASE3.is_absolute():
    PHASE3 = ROOT / PHASE3
if not PHASE1.is_absolute():
    PHASE1 = ROOT / PHASE1
print("Phase 1:", PHASE1)
print("Phase 3:", PHASE3)
print("Torch:", torch.__version__, "device: CPU")

## 1. Load the subject-grouped split

Notebook 01 fixes the literature-derived group assignments after cohort inspection. Check group isolation and cohort coverage before fitting the CNN. Original patient IDs remain unavailable.

In [ ]:
windows = pd.read_parquet(PHASE1 / "windows.parquet")
phase1_splits = pd.read_csv(PHASE1 / "splits.csv")
assert (
    windows[["record_id", "group_id", "split"]]
    .drop_duplicates()
    .sort_values("record_id")
    .reset_index(drop=True)
    .equals(phase1_splits.sort_values("record_id").reset_index(drop=True))
)
subject_splits = pd.read_csv(SPLIT_RUN / "subject_splits.csv")
assert set(subject_splits.record_id) == set(windows.record_id)
assert subject_splits.groupby("subject_group").split.nunique().max() == 1
display(subject_splits)
cohort = windows.drop(columns="split").merge(
    subject_splits[["record_id", "subject_group", "split"]], on="record_id", validate="many_to_one"
)
display(cohort.groupby(["split", "label"]).size().unstack(fill_value=0))
print("Subject groups:", subject_splits.subject_group.nunique())

## 2. Build one-minute waveform inputs

Use the Phase 1 filtered ECG samples. `resample_poly` reduces 100 Hz to 25 Hz, yielding 1,500 points per minute. Each minute is centered and divided by its own standard deviation (floor `1e-3` mV), then clipped to ±8. This deterministic, label-free normalization has no learned cohort statistics. All Phase 1 windows, including quality-flagged windows, remain in the comparison.

In [ ]:
def normalize_minute(minute, fs):
    minute = np.asarray(minute, dtype=float)
    if fs != 100 or minute.shape != (6000,) or not np.isfinite(minute).all():
        raise ValueError("Expected a finite 60-second ECG at 100 Hz (6000 samples)")
    return np.clip((minute - minute.mean()) / max(float(minute.std()), 1e-3), -8, 8).astype(
        "float32"
    )


def validate_cnn_cache(manifest, source_hash, phase1_hash=None):
    if (manifest.get("input_hz"), manifest.get("input_samples"), manifest.get("parameters")) != (
        100,
        6000,
        3009,
    ):
        raise ValueError("Incompatible CNN input/checkpoint metadata; choose a fresh run")
    if phase1_hash is not None and manifest.get("phase1_manifest_sha256") != phase1_hash:
        raise ValueError("CNN source data changed; choose a fresh run")
    if manifest.get("notebook_sha256") != source_hash or manifest.get("architecture") != str(
        make_model()
    ):
        raise ValueError("CNN source or architecture changed; choose a fresh run")


def build_waveform_array(cohort: pd.DataFrame, destination: Path) -> np.memmap:
    array = np.lib.format.open_memmap(
        destination, mode="w+", dtype="float32", shape=(len(cohort), WINDOW_SAMPLES)
    )
    for record_id, part in cohort.groupby("record_id", sort=True):
        with np.load(PHASE1 / "waveforms" / f"{record_id}.npz") as source:
            signal = source["filtered"]
            if int(source["fs"]) != FS_IN:
                raise ValueError(f"Unexpected sample rate: {record_id}")
            for row in part.itertuples():
                minute = signal[row.start_sample : row.end_sample]
                if len(minute) != FS_IN * 60 or not np.isfinite(minute).all():
                    raise ValueError(f"Invalid waveform: {record_id}/{row.minute}")
                normalized = normalize_minute(minute, int(source["fs"]))
                array[row.Index] = normalized.astype("float32")
    array.flush()
    return array


class WindowDataset(Dataset):
    def __init__(self, matrix: np.ndarray, labels: np.ndarray, indices: np.ndarray):
        self.matrix, self.labels, self.indices = matrix, labels, indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        index = int(self.indices[position])
        return torch.from_numpy(np.array(self.matrix[index : index + 1], copy=True)), torch.tensor(
            self.labels[index], dtype=torch.float32
        )


def make_model() -> nn.Module:
    return nn.Sequential(
        nn.Conv1d(1, 8, 15, stride=2, padding=7),
        nn.ReLU(),
        nn.MaxPool1d(2),
        nn.Conv1d(8, 16, 7, stride=2, padding=3),
        nn.ReLU(),
        nn.MaxPool1d(2),
        nn.Conv1d(16, 24, 5, stride=2, padding=2),
        nn.ReLU(),
        nn.AdaptiveAvgPool1d(1),
        nn.Flatten(),
        nn.Linear(24, 1),
    )


def choose_threshold(labels: np.ndarray, probability: np.ndarray) -> float:
    precision, recall, thresholds = precision_recall_curve(labels, probability)
    scores = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return float(thresholds[np.argmax(scores)])


def scores(labels: np.ndarray, probability: np.ndarray, threshold: float) -> dict:
    prediction = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, prediction, labels=[0, 1]).ravel()
    return dict(
        sensitivity=tp / (tp + fn),
        specificity=tn / (tn + fp),
        f1_score=f1_score(labels, prediction),
        pr_auc_average_precision=average_precision_score(labels, probability),
        roc_auc=roc_auc_score(labels, probability),
        accuracy=accuracy_score(labels, prediction),
        tn=int(tn),
        fp=int(fp),
        fn=int(fn),
        tp=int(tp),
    )


assert sum(p.numel() for p in make_model().parameters()) < 5000
assert np.isfinite(
    np.clip((np.zeros(6000) - 0) / max(float(np.zeros(6000).std()), 1e-3), -8, 8)
).all()

## 3. Train with validation-only selection

Train on subject-grouped training records, select the checkpoint with the lowest validation cross-entropy, then select a classification threshold maximizing validation F1-score (lowest threshold on a tie). The test data are evaluated only after those choices are frozen. Set `APNEA_PHASE3_RUN` to a **new** directory to rerun; an existing run is read, never overwritten.

In [ ]:
def predict(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    chunks = []
    with torch.inference_mode():
        for x, _ in loader:
            chunks.append(torch.sigmoid(model(x).squeeze(1)).numpy())
    return np.concatenate(chunks)


def run_cnn(cohort: pd.DataFrame, destination: Path) -> None:
    if destination.exists():
        raise FileExistsError(f"Choose a new output directory: {destination}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    with tempfile.TemporaryDirectory(prefix=".phase3-", dir=destination.parent) as temporary:
        stage = Path(temporary) / "run"
        stage.mkdir()
        subject_splits.to_csv(stage / "subject_splits.csv", index=False)
        matrix = build_waveform_array(cohort.reset_index(drop=True), stage / "waveforms.npy")
        labels = cohort.label.to_numpy(dtype="float32")
        indices = {
            split: np.flatnonzero(cohort.split.to_numpy() == split)
            for split in ("train", "validation", "test")
        }
        loaders = {
            split: DataLoader(
                WindowDataset(matrix, labels, idx),
                batch_size=128,
                shuffle=split == "train",
                num_workers=0,
            )
            for split, idx in indices.items()
        }
        model = make_model()
        train_labels = labels[indices["train"]]
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(float((train_labels == 0).sum() / (train_labels == 1).sum()))
        )
        validation_criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        best_loss, stale, history = float("inf"), 0, []
        for epoch in range(1, 13):
            model.train()
            train_sum = 0.0
            for x, y in loaders["train"]:
                optimizer.zero_grad(set_to_none=True)
                logits = model(x).squeeze(1)
                loss = criterion(logits, y)
                loss.backward()
                optimizer.step()
                train_sum += float(loss.item()) * len(y)
            model.eval()
            validation_sum = 0.0
            with torch.inference_mode():
                for x, y in loaders["validation"]:
                    validation_sum += float(
                        validation_criterion(model(x).squeeze(1), y).item()
                    ) * len(y)
            train_loss = train_sum / len(indices["train"])
            validation_loss = validation_sum / len(indices["validation"])
            history.append(
                dict(epoch=epoch, train_loss=train_loss, validation_loss=validation_loss)
            )
            print(
                f"Epoch {epoch:02d}: train={train_loss:.4f}, validation={validation_loss:.4f}",
                flush=True,
            )
            if validation_loss < best_loss - 1e-5:
                best_loss, stale = validation_loss, 0
                torch.save(model.state_dict(), stage / "cnn.pt")
            else:
                stale += 1
                if stale >= 3:
                    break
        model.load_state_dict(torch.load(stage / "cnn.pt", map_location="cpu", weights_only=True))
        pd.DataFrame(history).to_csv(stage / "training_history.csv", index=False)
        probabilities = {split: predict(model, loaders[split]) for split in ("validation", "test")}
        threshold = choose_threshold(
            labels[indices["validation"]].astype(int), probabilities["validation"]
        )
        metrics, predictions = [], []
        for split in ("validation", "test"):
            idx = indices[split]
            truth = labels[idx].astype(int)
            probability = probabilities[split]
            metrics.append(
                dict(
                    model="cnn",
                    feature_set="waveform",
                    partition=split,
                    threshold=threshold,
                    n_windows=len(idx),
                    **scores(truth, probability, threshold),
                )
            )
            detail = cohort.iloc[idx][
                ["record_id", "minute", "label", "split", "quality_flag"]
            ].copy()
            detail["model"] = "cnn"
            detail["feature_set"] = "waveform"
            detail["probability"] = probability
            detail["prediction"] = (probability >= threshold).astype(int)
            predictions.append(detail)
        pd.DataFrame(metrics).to_csv(stage / "metrics.csv", index=False)
        pd.concat(predictions, ignore_index=True).to_parquet(
            stage / "predictions.parquet", index=False
        )
        (stage / "manifest.json").write_text(
            json.dumps(
                dict(
                    created_utc=datetime.now(UTC).isoformat(),
                    seed=SEED,
                    phase1_manifest_sha256=hashlib.sha256(
                        (PHASE1 / "manifest.json").read_bytes()
                    ).hexdigest(),
                    split_method=(
                        "StratifiedGroupKFold 5 folds; fold 0 test, 1 validation, 2-4 train"
                    ),
                    subject_group_source=(
                        "Yeh et al. Sensors 2022 Table 1; c05/c06 overlap documented by PhysioNet"
                    ),
                    original_patient_ids_available=False,
                    n_windows=len(cohort),
                    n_subject_groups=subject_splits.subject_group.nunique(),
                    model=(
                        "Conv1d(1,8,15)-Conv1d(8,16,7)-Conv1d(16,24,5)-global average pool-linear"
                    ),
                    parameters=sum(p.numel() for p in model.parameters()),
                    notebook_sha256=hashlib.sha256(
                        (ROOT / "notebooks/03_waveform_cnn.ipynb").read_bytes()
                    ).hexdigest(),
                    architecture=str(make_model()),
                    checkpoint_sha256=hashlib.sha256((stage / "cnn.pt").read_bytes()).hexdigest(),
                    input_hz=FS_MODEL,
                    input_samples=WINDOW_SAMPLES,
                    waveform_policy=(
                        "Phase 1 filtered ECG; 100 Hz, 6000 samples, no input resampling; "
                        "within-window z-score with 1e-3 mV floor; clip +/-8"
                    ),
                    checkpoint_rule=(
                        "lowest validation unweighted BCE loss; "
                        "early stop after 3 stale epochs; at most 12 epochs"
                    ),
                    threshold_rule="maximize validation F1-score; lowest threshold on ties",
                    pr_auc_definition="scikit-learn average_precision_score",
                    inference_scope="exploratory; prior Phase 2 test results inspected",
                ),
                indent=2,
            )
            + "\n"
        )
        del matrix
        stage.rename(destination)


if PHASE3.exists():
    saved_manifest = json.loads((PHASE3 / "manifest.json").read_text())
    validate_cnn_cache(
        saved_manifest,
        hashlib.sha256((ROOT / "notebooks/03_waveform_cnn.ipynb").read_bytes()).hexdigest(),
        hashlib.sha256((PHASE1 / "manifest.json").read_bytes()).hexdigest(),
    )
    if (
        hashlib.sha256((PHASE3 / "cnn.pt").read_bytes()).hexdigest()
        != saved_manifest["checkpoint_sha256"]
    ):
        raise ValueError("CNN checkpoint checksum mismatch")
    if np.load(PHASE3 / "waveforms.npy", mmap_mode="r").shape != (len(cohort), 6000):
        raise ValueError("Cached waveform shape mismatch")
    print("Reusing verified 100 Hz Phase 3 run:", PHASE3)
else:
    run_cnn(cohort, PHASE3)
metrics = pd.read_csv(PHASE3 / "metrics.csv")
predictions = pd.read_parquet(PHASE3 / "predictions.parquet")
manifest = json.loads((PHASE3 / "manifest.json").read_text())
assert len(metrics) == 2 and len(predictions) == len(cohort[cohort.split != "train"])
assert pd.read_csv(PHASE3 / "subject_splits.csv").equals(subject_splits)
display(metrics.round(3))

## 4. Inspect learning and errors

Plot training and validation loss, then count false positives and false negatives by recording. These are diagnostic descriptions only; they do not change the frozen checkpoint or threshold.

In [ ]:
history = pd.read_csv(PHASE3 / "training_history.csv")
fig, ax = plt.subplots(figsize=(7, 4))
history.plot(x="epoch", y=["train_loss", "validation_loss"], ax=ax, marker="o")
ax.set_ylabel("Binary cross-entropy")
display(fig)
plt.close(fig)
test_errors = predictions[predictions.split == "test"].copy()
test_errors["outcome"] = np.select(
    [
        test_errors.label.eq(1) & test_errors.prediction.eq(1),
        test_errors.label.eq(0) & test_errors.prediction.eq(0),
        test_errors.label.eq(0) & test_errors.prediction.eq(1),
    ],
    ["TP", "TN", "FP"],
    default="FN",
)
display(test_errors.groupby(["record_id", "outcome"]).size().unstack(fill_value=0))

## Interpretation

The source-matched groups are stronger than recording-only partitions, but original patient IDs are unavailable. A small number of grouped subjects and the previously inspected Phase 2 test results limit inference. Raw waveform classification here is retrospective and has no clinical diagnostic validation. Continue with [04_optimization_evaluation.ipynb](04_optimization_evaluation.ipynb).